# Neuro MVP 6 — Phase 6B Demo (Colab T4 GPU)

This notebook runs the **Phase 6B** reaching experiment end-to-end on a
Colab **T4 GPU** runtime and renders an MP4 of the brain reaching to a
target using a 2-link MuJoCo arm simulated with MJX.

**Pipeline**
1. Install `mujoco`, `mujoco-mjx`, `equinox`, `imageio[ffmpeg]`.
2. Upload or clone the `Neuro_MVP_6` repo.
3. Confirm JAX sees the T4 GPU.
4. Build the reacher (`build_reacher`) — brain + MjxArmBody.
5. Motor **babbling** (30 000 cycles, OU exploration).
6. **Reaching** training (2000 episodes).
7. **Sleep cycle** (SWS replay + REM rollout).
8. Render a highlight reach episode to an MP4 video and display it inline.

> Requires **Runtime → Change runtime type → T4 GPU**.

## 1. Install dependencies

In [2]:
!pip -q install "jax[cuda12]" equinox mujoco mujoco-mjx "imageio[ffmpeg]" numpy

In [3]:
!nvidia-smi

Tue Apr 28 19:29:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get the repo
Either upload the project as a zip, clone from GitHub, or mount Drive.

In [4]:
# Option A: upload a zip of the repo through the file browser then unzip:
# !unzip -q Neuro_MVP_6.zip -d /content/
# %cd /content/Neuro_MVP_6

# Option B: clone from GitHub (adjust URL):
# !git clone https://github.com/<you>/Neuro_MVP_6.git /content/Neuro_MVP_6
# %cd /content/Neuro_MVP_6

# Option C: mount Google Drive:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/SNN/Neuro_MVP_6

import os, sys
sys.path.insert(0, os.getcwd())
print('CWD =', os.getcwd())

Mounted at /content/drive
/content/drive/MyDrive/SNN/Neuro_MVP_6
CWD = /content/drive/MyDrive/SNN/Neuro_MVP_6


## 3. JAX GPU sanity check

In [5]:
import jax
print('JAX backend:', jax.default_backend())
print('Devices :', jax.devices())
assert 'gpu' in jax.default_backend() or jax.default_backend() == 'cuda', (
    'T4 GPU not detected — use Runtime → Change runtime type → T4 GPU.'
)

JAX backend: gpu
Devices : [CudaDevice(id=0)]


## 4. Build the reacher (brain + MjxArmBody)

In [6]:
import time
import warnings
import numpy as np
import jax
import jax.numpy as jnp

from core.backend import DEFAULT, make_key
from embodiment.reacher_env import build_reacher
from embodiment.mjx_run_loop import (
    run_reach_episode, run_babbling, collect_render_frames,
)

# Treat the "JAX array set as static" warning as an error
warnings.filterwarnings("error", message=".*JAX array is being set as static.*")

params, state, body = build_reacher(make_key(0))

warnings.resetwarnings()

print('brain_state OK — cortex L5 =', state.cortex.rate_l5.shape)
print('body joints=', body.cfg.n_joints, 'motor_dim=', body.cfg.motor_dim)


Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
brain_state OK — cortex L5 = (32,)
body joints= 2 motor_dim= 2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 5. Motor babbling (30 000 cycles)

OU-noise-driven exploration of the arm workspace to train the cerebellum
forward model and the cortical state embedding.

### 5a. Physics-only diagnostic

Isolates `mjx.step` cost from the rest of the pipeline.  After the
`<option>` rewrite (`integrator="implicitfast"`, `solver="CG"`,
`iterations="2"`, contacts disabled), a single `mjx.step` on this
2-DOF arm should run in **~1–3 ms** on a T4.  If this cell reports
> 10 ms / step, the solver config did not take effect (check that
the kernel was restarted after editing `mjx_arm_body.py`).

In [7]:
from mujoco import mjx
import equinox as eqx

# Isolate the physics step: JIT a pure N-step rollout with no brain,
# no OU noise, no sensory encoding.  Whatever time-per-step this
# reports is a *lower bound* on any `frame_skip` cost.
_mjx_model = body.mjx_model
_mjx_data0 = body.mjx_data

@eqx.filter_jit
def _pure_mjx_scan(data0, ctrl, n_steps):
    def step(d, _):
        return mjx.step(_mjx_model, d.replace(ctrl=ctrl)), None
    final, _ = jax.lax.scan(step, data0, xs=None, length=n_steps)
    return final

# Warm-up: compile + 1 call.
ctrl = jnp.array([0.3, -0.3], jnp.float32)
N = 2000
_ = _pure_mjx_scan(_mjx_data0, ctrl, N)
jax.block_until_ready(_.qpos)

# Measure steady-state.
t0 = time.time()
out = _pure_mjx_scan(_mjx_data0, ctrl, N)
jax.block_until_ready(out.qpos)
dt = time.time() - t0
us_per_step = dt / N * 1e6
print(f"pure mjx.step (N={N}): {dt*1000:.1f} ms total → {us_per_step:.1f} µs / step")
if us_per_step > 10_000:
    print("  ⚠  > 10 ms / mjx.step — solver config likely NOT applied.")
    print("     Check Runtime → Restart runtime after editing mjx_arm_body.py.")
elif us_per_step < 5_000:
    print(f"  ✔  At frame_skip=3 this contributes ~{us_per_step*3/1000:.1f} ms / cycle.")

# Sanity: final qpos should be a reasonable 2-vector, not NaN / huge.
print("final qpos[:2]:", np.asarray(out.qpos[:2]))


pure mjx.step (N=2000): 465.5 ms total → 232.8 µs / step
  ✔  At frame_skip=5 this contributes ~0.7 ms / cycle.
final qpos[:2]: [ 0.29999974 -0.2999997 ]


In [8]:
# After code changes: Runtime → Restart runtime (this reloads
# embodiment.mjx_run_loop cleanly — do NOT importlib.reload it).
#
# Implementation status:
#   * ``embodiment/mjx_arm_body.py``  — physics ``frame_skip`` now uses
#     ``jax.lax.fori_loop``  (1 compiled ``mjx.step``, not ``frame_skip``).
#   * ``core/brain_graph.py``         — both ``params.substeps`` Python
#     loops replaced by ``jax.lax.scan``  (cortex/thalamus/BG graph
#     traced ONCE, not ``substeps``×).
#   * ``embodiment/mjx_run_loop.py``  — outer driver already wraps the
#     whole cycle in ``jax.lax.scan`` inside one ``@eqx.filter_jit``.
#
# Result: ONE XLA compile per chunk-length, regardless of n_cycles.
# Second identical-sized chunk reuses the cached executable.

CHUNK = 500  # canonical chunk size — keep constant across probes
             # so the second probe hits the cache.

print(f"=== Probe 1 (cold): {CHUNK} cycles — pays compile cost ===")
t0 = time.time()
bab1 = run_babbling(
    state, params, DEFAULT, body, jax.random.PRNGKey(0),
    n_cycles=CHUNK, target_refresh=CHUNK,
)
bab1.tip_traj.block_until_ready()
dt_cold = time.time() - t0
print(f"  cold : {dt_cold:6.2f}s  ({dt_cold/CHUNK*1000:.1f} ms / cycle)")

print(f"=== Probe 2 (warm): {CHUNK} cycles — cache hit, steady state ===")
t0 = time.time()
bab2 = run_babbling(
    bab1.brain_state, params, DEFAULT, bab1.body, jax.random.PRNGKey(1),
    n_cycles=CHUNK, target_refresh=CHUNK,
)
bab2.tip_traj.block_until_ready()
dt_warm = time.time() - t0
print(f"  warm : {dt_warm:6.2f}s  ({dt_warm/CHUNK*1000:.1f} ms / cycle)")

# ---- Full babbling run
t0 = time.time()
bab = run_babbling(
    bab2.brain_state, params, DEFAULT, bab2.body, make_key(2),
    n_cycles=30_000, ou_tau=20.0, ou_sigma=0.4,
    target_refresh=CHUNK,  # SAME chunk length — no recompile
)
state, body = bab.brain_state, bab.body
print(f'babbling: {time.time()-t0:.1f}s for 30k cycles')
tips = np.asarray(bab.tip_traj)
print('tip bbox:', tips.min(0), tips.max(0))


=== Probe 1 (cold): 500 cycles — pays compile cost ===
  cold :  38.51s  (77.0 ms / cycle)
=== Probe 2 (warm): 500 cycles — cache hit, steady state ===
  warm :  34.34s  (68.7 ms / cycle)
babbling: 231.7s for 30k cycles
tip bbox: [-0.28924417  0.03879373] [0.49587673 0.43733245]


### 5c. Babbling-coverage diagnostic

After the OU-discretisation + ctrlrange fixes, the 30 k babble should:

* **OU command std ≈ 0.4** (was ~0.06 before the fix).
* **Joint qpos** should reach a wide slice of ±2 rad (was clipped at ±1).
* **Tip bbox** should cover roughly `x ∈ [-0.3, 0.5]`, `y ∈ [-0.45, 0.45]`
  — most of the reachable crescent — instead of a 2 cm band.


In [9]:
# Run this AFTER cell 5's full 30 k babble has finished.  Uses the
# recorded OU joint-command trajectory (`bab.jc_traj`) and tip trace
# to verify exploration is healthy before spending 77 min on cell 6.
import numpy as np

jcs  = np.asarray(bab.jc_traj)          # (n_cycles, motor_dim)  in [-1, 1]
tips = np.asarray(bab.tip_traj)         # (n_cycles, 2)          metres

# 1) OU sanity: steady-state std should be ~ou_sigma = 0.4
print(f"jc per-channel mean: {jcs.mean(axis=0)}")
print(f"jc per-channel std : {jcs.std(axis=0)}  (target ~0.4)")
print(f"jc |value| max     : {np.abs(jcs).max():.3f}  (target close to 1.0)")

# 2) Tip coverage: should be a wide crescent, not a narrow band
xmin, ymin = tips.min(axis=0)
xmax, ymax = tips.max(axis=0)
print(f"tip x range: [{xmin:+.3f}, {xmax:+.3f}]  span={xmax-xmin:.3f} m")
print(f"tip y range: [{ymin:+.3f}, {ymax:+.3f}]  span={ymax-ymin:.3f} m")

# 3) Warn on known failure modes
if jcs.std(axis=0).max() < 0.2:
    print("  ⚠  OU std still too small — check ou_babble_step fix.")
if (xmax - xmin) < 0.15 or (ymax - ymin) < 0.15:
    print("  ⚠  Tip still stuck in a narrow band — check ctrlrange / joint_range rescale.")
else:
    print("  ✔  Workspace coverage looks healthy — OK to run cell 6.")


jc per-channel mean: [0.9995429 0.9895931]
jc per-channel std : [0.00095937 0.03104619]  (target ~0.4)
jc |value| max     : 1.000  (target close to 1.0)
tip x range: [-0.289, +0.496]  span=0.785 m
tip y range: [+0.039, +0.437]  span=0.399 m
  ⚠  OU std still too small — check ou_babble_step fix.
  ✔  Workspace coverage looks healthy — OK to run cell 6.


## 6. Reaching (2 000 episodes)

In [10]:
N_EPISODES = 2000
MAX_STEPS  = 300
success_curve = []
mean_dist_curve = []
key = make_key(2)
t0 = time.time()
for ep in range(N_EPISODES):
    key, k_ep = jax.random.split(key)
    res = run_reach_episode(
        state, params, DEFAULT, body, k_ep,
        max_steps=MAX_STEPS, reset_body=True,
    )
    state, body = res.brain_state, res.body
    success_curve.append(bool(res.success))
    mean_dist_curve.append(float(jnp.mean(res.dists[-50:])))
    if (ep + 1) % 50 == 0:
        recent = np.mean(success_curve[-50:])
        print(f'ep {ep+1:4d}/{N_EPISODES}  success@50={recent:.2f}  '
              f'meanD={np.mean(mean_dist_curve[-50:]):.3f}  '
              f'({time.time()-t0:.0f}s)')
print('training success rate (last 200):',
      np.mean(success_curve[-200:]))

ep   50/2000  success@50=0.06  meanD=0.375  (138s)
ep  100/2000  success@50=0.08  meanD=0.375  (239s)
ep  150/2000  success@50=0.10  meanD=0.395  (341s)
ep  200/2000  success@50=0.12  meanD=0.358  (442s)
ep  250/2000  success@50=0.08  meanD=0.400  (542s)
ep  300/2000  success@50=0.12  meanD=0.324  (643s)
ep  350/2000  success@50=0.04  meanD=0.369  (743s)
ep  400/2000  success@50=0.08  meanD=0.376  (844s)
ep  450/2000  success@50=0.02  meanD=0.383  (957s)
ep  500/2000  success@50=0.12  meanD=0.368  (1070s)
ep  550/2000  success@50=0.08  meanD=0.399  (1183s)
ep  600/2000  success@50=0.06  meanD=0.372  (1294s)
ep  650/2000  success@50=0.16  meanD=0.411  (1403s)
ep  700/2000  success@50=0.18  meanD=0.383  (1516s)
ep  750/2000  success@50=0.06  meanD=0.397  (1633s)
ep  800/2000  success@50=0.08  meanD=0.422  (1748s)
ep  850/2000  success@50=0.14  meanD=0.376  (1887s)
ep  900/2000  success@50=0.10  meanD=0.330  (1987s)
ep  950/2000  success@50=0.02  meanD=0.409  (2087s)
ep 1000/2000  success

In [ ]:
ADDITIONAL_EPISODES = 3000
PREV_EPISODES = 2000  # Zmienna pomocnicza do poprawnego wyświetlania logów

print(f"Rozpoczynam kontynuację treningu od epizodu {PREV_EPISODES}...")
t0 = time.time()

for ep in range(ADDITIONAL_EPISODES):
    # Używamy klucza 'key', który został zaktualizowany na końcu poprzedniej komórki
    key, k_ep = jax.random.split(key)
    
    # Przekazujemy 'state' i 'params' zawierające wiedzę z pierwszych 2000 epizodów
    res = run_reach_episode(
        state, params, DEFAULT, body, k_ep,
        max_steps=MAX_STEPS, reset_body=True,
    )
    
    # Krytyczne: nadpisujemy stan, aby zachować ciągłość modyfikacji wag
    state, body = res.brain_state, res.body
    
    # Kontynuujemy dodawanie danych do istniejących list
    success_curve.append(bool(res.success))
    mean_dist_curve.append(float(jnp.mean(res.dists[-50:])))
    
    if (ep + 1) % 100 == 0:
        recent = np.mean(success_curve[-100:])
        current_ep = PREV_EPISODES + ep + 1
        print(f'ep {current_ep:4d}  success@100={recent:.2f}  '
              f'meanD={np.mean(mean_dist_curve[-100:]):.3f}  '
              f'({time.time()-t0:.0f}s)')
              
print('training success rate (last 200):', np.mean(success_curve[-200:]))

## 7. Sleep cycle (SWS + REM via `brain_cycle`)

In [ ]:
from core.brain_graph import brain_cycle
pre_success = np.mean(success_curve[-200:])
# Force several sleep cycles by invoking brain_cycle with a dim, static
# scene so the endogenous sleep switch (Phase 5B) can trigger (low
# retinal drive → adenosine accumulation → SWS).  brain_cycle takes
# `image` + `fixation_xy`, not `sensory` (that is an internal signal
# computed by the retina/LGN/V1 chain inside action_brain_step).
key = make_key(7)
# 32x32 dim grey image + central fixation.  The retina input
# size is inferred by the sensory-stack params; 32x32 matches the
# default Phase 0 sensory-stack config.
dim_image  = jnp.full((32, 32), 0.05, jnp.float32)
center_fix = jnp.array([0.5, 0.5], jnp.float32)
for _ in range(300):
    key, k = jax.random.split(key)
    out = brain_cycle(
        state, params, DEFAULT,
        image=dim_image,
        fixation_xy=center_fix,
        prev_reward=jnp.asarray(0.0, jnp.float32),
        prev_done=jnp.asarray(0.0, jnp.float32),
        key=k,
    )
    state = out.state
print('mean ATP after sleep:',
      float(jnp.mean(state.astrocyte.atp)))
print('final sleep phase   :', int(state.sleep.phase))


In [ ]:
post_success = []
post_dists = []
key = make_key(3)
for ep in range(100):
    key, k_ep = jax.random.split(key)
    res = run_reach_episode(
        state, params, DEFAULT, body, k_ep,
        max_steps=MAX_STEPS, reset_body=True,
    )
    state, body = res.brain_state, res.body
    post_success.append(bool(res.success))
    post_dists.append(float(jnp.mean(res.dists[-50:])))
print(f'pre-sleep success:  {pre_success:.3f}')
print(f'post-sleep success: {np.mean(post_success):.3f}')
print(f'post-sleep meanD:   {np.mean(post_dists):.3f}')

## 8. Highlight reach episode → MP4 video

We capture the full trajectory of one reach, render each frame on CPU
via `mujoco.Renderer`, and save it to `reach_highlight.mp4`.

In [ ]:
highlight = run_reach_episode(
    state, params, DEFAULT, body, make_key(4242),
    max_steps=300, reset_body=True, record_qpos=True,
)
print('highlight success =', bool(highlight.success),
      ' final d =', float(highlight.dists[-1]))
print('qpos traj shape   =', highlight.qpos_traj.shape)

In [ ]:
import imageio
frames = collect_render_frames(
    body,
    highlight.qpos_traj,
    highlight.target_traj,
    width=480, height=480, camera='topdown',
)
print('rendered', len(frames), 'frames')
imageio.mimsave('reach_highlight.mp4', frames, fps=30, quality=7)
print('saved reach_highlight.mp4')

In [ ]:
import base64, pathlib
from IPython.display import HTML
mp4 = pathlib.Path('reach_highlight.mp4').read_bytes()
b64 = base64.b64encode(mp4).decode('ascii')
HTML(f'<video width=480 controls autoplay loop>'
     f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>')

---
**Done.**  The video above shows a biologically-grounded action brain
(cortex/thalamus/BG/cerebellum/M1/proprio) reaching to a target using
an MJX-simulated 2-link arm, with no backpropagation anywhere in the
learning pipeline.